# mount drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# importing libraries

In [1]:
import pandas as pd
import numpy as np
import random
import kagglehub
import os
import json
import joblib
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from scipy.signal import savgol_filter

c:\Users\ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
c:\Users\ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Project Paths

In [3]:
BASE_PATH = "../Data/"
RAW_PATH = os.path.join(BASE_PATH, "raw_data")
SAVE_PATH = os.path.join(BASE_PATH, "swat_preprocessed")

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(RAW_PATH, exist_ok=True)

print("raw path:", RAW_PATH)
print("Save path:", SAVE_PATH)

raw path: ../Data/raw_data
Save path: ../Data/swat_preprocessed


# Configuration

In [4]:
CONFIG = {
    "stabilization_hours": 5,
    "smoothing_window": 5,
    "train_ratio": 0.9,
    "window_size": 60,
    "stride_train": 1,
    "stride_val": 1,
    "stride_test": 1
}

# Load SWaT Data

In [11]:

ab109316_ts_training_path = kagglehub.dataset_download('ab109316/ts-training')
print('Data source import complete.')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df_attack_28_2 = pd.read_csv(ab109316_ts_training_path+"/TS Training/data/anomaly_detection/multivariate/SWAT/SWaT.csv")
#df_normal_v0 = pd.read_csv(ab109316_ts_training_path+"/TS Training/data/anomaly_detection/multivariate/SWAT/SWaT_Dataset_Normal_v0.csv")
#df_normal_v1=pd.read_csv(ab109316_ts_training_path+"/TS Training/data/anomaly_detection/multivariate/SWAT/SWaT_Dataset_Normal_v1.csv")
#df_normal = pd.concat([df_normal_v0, df_normal_v1]).sort_index()
df_normal=pd.read_csv(ab109316_ts_training_path+"/TS Training/data/anomaly_detection/multivariate/SWAT/SWaT_Dataset_Normal_v1.csv")

df_attack_28_2.to_csv('../Data/raw_data/df_attack.csv')
df_normal.to_csv('../Data/raw_data/df_normal.csv')

Data source import complete.


In [9]:
def load_swat(normal_path, attack_path):

    normal = pd.read_csv(normal_path, low_memory=False)
    attack = pd.read_csv(attack_path, low_memory=False)

    for df in [normal, attack]:
        df.columns = df.columns.str.strip()

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"].str.strip(),
            format="%d/%m/%Y %I:%M:%S %p",
            errors="coerce"
        )

        df.sort_values("Timestamp", inplace=True)
        df.set_index("Timestamp", inplace=True)

        df["label"] = df["Normal/Attack"].map(
            lambda x: 0 if x == "Normal" else 1
        )

        df.drop(columns=["Normal/Attack"], inplace=True)

    return normal, attack




In [12]:
NORMAL_PATH = os.path.join(RAW_PATH, "df_normal.csv")
ATTACK_PATH = os.path.join(RAW_PATH, "df_attack.csv")

normal_raw, attack_raw = load_swat(NORMAL_PATH, ATTACK_PATH)

print("Normal shape:", normal_raw.shape)
print("Attack shape:", attack_raw.shape)

Normal shape: (495000, 53)
Attack shape: (449919, 53)


# cleaning data

## Remove Stabilization Period

In [13]:
def remove_stabilization(df, hours):
    cutoff = df.index[0] + pd.Timedelta(hours=hours)
    return df[df.index > cutoff]

normal = remove_stabilization(normal_raw, CONFIG["stabilization_hours"])
attack = attack_raw.copy()

print("Normal after stabilization:", normal.shape)

Normal after stabilization: (476999, 53)


## Remove Constant Features

In [14]:
def drop_common_constant_columns(df1, df2, label_col="label"):
    const_df1 = set(c for c in df1.columns
                    if df1[c].nunique() <= 1 and c != label_col)
    const_df2 = set(c for c in df2.columns
                    if df2[c].nunique() <= 1 and c != label_col)

    common_constant_cols = list(const_df1.intersection(const_df2))

    df1 = df1.drop(columns=common_constant_cols)
    df2 = df2.drop(columns=common_constant_cols)

    return df1, df2, common_constant_cols

normal, attack, dropped_cols = drop_common_constant_columns(normal, attack)

print("Dropped constant columns:", dropped_cols)

Dropped constant columns: ['P202', 'P404', 'P502', 'P603', 'P401', 'P601']


## Fix Timestamp Gaps

In [15]:
def fix_gaps(df):
    df = df.groupby(df.index).mean()

    full_range = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq="s"
    )

    df = df.reindex(full_range)
    return df

normal = fix_gaps(normal)
attack = fix_gaps(attack)

In [16]:
normal.shape

(476999, 47)

# feature engineering

## Identify Sensor Columns

In [17]:
feature_cols = [c for c in normal.columns if c != "label"]

## Smooth Sensors (Rolling Average)

In [18]:
# ============================================================
# Optional signal smoothing
# ============================================================

def smooth_signals(df, feature_cols, window=5, enable=True):

    if not enable:
        print("Skipping smoothing.")
        return df

    df = df.copy()

    df[feature_cols] = (
        df[feature_cols]
        .rolling(window=window, min_periods=1)
        .mean()
    )

    print(f"Smoothing applied with window={window}")

    return df


In [19]:
SMOOTHING_ENABLED = False

normal = smooth_signals(normal, feature_cols, CONFIG["smoothing_window"], SMOOTHING_ENABLED)
attack = smooth_signals(attack, feature_cols, CONFIG["smoothing_window"], SMOOTHING_ENABLED)

Skipping smoothing.
Skipping smoothing.


## Proper Train / Validation Split

In [17]:

# attack_only = attack[attack["label"] == 1]
# attack_normal = attack[attack["label"] == 0]

# n_attack = len(attack_only)
# attack_normal_keep = attack_normal.sample(n=n_attack, random_state=42)

# attack_normal_for_train = attack_normal.drop(attack_normal_keep.index)

# normal = pd.concat([normal, attack_normal_for_train]).sort_index()

# attack = pd.concat([attack_only, attack_normal_keep]).sort_index()

In [20]:
split_idx = int(len(normal) * CONFIG["train_ratio"])

normal_train = normal.iloc[:split_idx]
normal_val   = normal.iloc[split_idx:]

print("Train shape:", normal_train.shape)
print("Val shape:", normal_val.shape)

Train shape: (429299, 47)
Val shape: (47700, 47)


## Fit Scaler ONLY On Train

In [21]:
scaler = StandardScaler()
scaler.fit(normal_train[feature_cols])

,copy,True
,with_mean,True
,with_std,True


## Apply Scaling

In [22]:

def scale(df, scaler, feature_cols):
    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.transform(df[feature_cols])
    return df_scaled

normal_train = scale(normal_train, scaler, feature_cols)
normal_val   = scale(normal_val, scaler, feature_cols)
attack       = scale(attack, scaler, feature_cols)

## Window Function

In [23]:
def create_windows(data, labels, window_size, stride):

    n_samples = len(data)
    n_features = data.shape[1]

    n_windows = (n_samples - window_size) // stride + 1

    X = np.zeros((n_windows, window_size, n_features), dtype=np.float32)
    y = np.zeros(n_windows, dtype=np.int8)

    for i, start in enumerate(
        range(0, n_samples - window_size + 1, stride)
    ):
        end = start + window_size
        X[i] = data[start:end]
        y[i] = 1 if labels[start:end].max() > 0 else 0

    return X, y

## Create Windows

In [24]:
WINDOW = CONFIG["window_size"]

X_train, y_train = create_windows(
    normal_train[feature_cols].values,
    normal_train["label"].values,
    WINDOW,
    CONFIG["stride_train"]
)

X_val, y_val = create_windows(
    normal_val[feature_cols].values,
    normal_val["label"].values,
    WINDOW,
    CONFIG["stride_val"]
)

X_attack, y_attack = create_windows(
    attack[feature_cols].values,
    attack["label"].values,
    WINDOW,
    CONFIG["stride_test"]
)

print("Train windows:", X_train.shape)
print("Val windows:", X_val.shape)
print("Attack windows:", X_attack.shape)

Train windows: (429240, 60, 46)
Val windows: (47641, 60, 46)
Attack windows: (449941, 60, 46)


In [25]:
# ============================================================
# Dataset summary
# ============================================================

print("\nDataset shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_attack:", X_attack.shape)
print("y_attack:", y_attack.shape)


Dataset shapes:
X_train: (429240, 60, 46)
y_train: (429240,)
X_val: (47641, 60, 46)
y_val: (47641,)
X_attack: (449941, 60, 46)
y_attack: (449941,)


## Save Everything to Drive

In [26]:
np.save(os.path.join(SAVE_PATH, "X_train.npy"), X_train)
np.save(os.path.join(SAVE_PATH, "y_train.npy"), y_train)

np.save(os.path.join(SAVE_PATH, "X_val.npy"), X_val)
np.save(os.path.join(SAVE_PATH, "y_val.npy"), y_val)

np.save(os.path.join(SAVE_PATH, "X_attack.npy"), X_attack)
np.save(os.path.join(SAVE_PATH, "y_attack.npy"), y_attack)

joblib.dump(scaler, os.path.join(SAVE_PATH, "scaler.pkl"))

with open(os.path.join(SAVE_PATH, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

print("All files saved to:", SAVE_PATH)

All files saved to: ../Data/swat_preprocessed


In [27]:
with open(os.path.join(SAVE_PATH, "feature_columns.json"), "w") as f:
    json.dump(feature_cols, f, indent=2)

print(f"Saved {len(feature_cols)} feature names to {os.path.join(SAVE_PATH, 'feature_columns.json')}.")

Saved 46 feature names to ../Data/swat_preprocessed\feature_columns.json.
